In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image 

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
import mlflow
import src.config as config
from src.mlflow_init import init_mlflow

init_mlflow(config.EXPERIMENT_NAME)

EPOCHS = config.EPOCHS
BATCH_SIZE = config.BATCH_SIZE
RANDOM_SEED = config.RANDOM_SEED
IMAGE_SIZE = config.IMAGE_SIZE


In [ ]:
LABELS_PATH = "../dataset/labels/labels.csv"

df = pd.read_csv(LABELS_PATH)

df.head()

In [ ]:
print(df.shape)
print(df.head())

In [ ]:
from src.label_mapping import create_label_mapping, save_label_mapping

label_to_idx, idx_to_label = create_label_mapping(r"../dataset/labels/labels.csv")
save_label_mapping(label_to_idx, idx_to_label, "../artifacts")

NUM_CLASSES = len(label_to_idx)
print(NUM_CLASSES)

In [ ]:
label_to_idx

In [ ]:
# Mappings are saved centrally in artifacts/ directory. No local generation needed.

In [ ]:
IMAGE_SIZE = 256

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.ToTensor(),
])

In [ ]:
class TankriDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_dir,
        transform=None
    ):

        self.dataframe = dataframe

        self.image_dir = image_dir

        self.transform = transform

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, idx):

        image_name = self.dataframe.iloc[idx]["image"]

        label = self.dataframe.iloc[idx]["label"]

        image_path = os.path.join(
            self.image_dir,
            image_name
        )

        image = Image.open(image_path)

        if self.transform:
            image = self.transform(image)

        label = label_to_idx[label]

        return image, label

In [ ]:
dataset = TankriDataset(
    dataframe=df,
    image_dir="../dataset/images",
    transform=transform
)
image, label = dataset[0]

plt.imshow(
    image.squeeze(),
    cmap="gray"
)

plt.title(idx_to_label[label])

plt.axis("off")

plt.show()

In [ ]:
image, label = dataset[0]

print(type(image))
print(image.dtype)
print(image.shape)
print(label)
print(idx_to_label[label]) 
print(image.min())
print(image.max())

In [ ]:
print(image[:, :5, :5])

In [ ]:
batch_size = config.BATCH_SIZE

train_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True
)


In [ ]:
counts = df["label"].value_counts()

print(counts.sort_values())
print(counts[counts == 1])
print(counts[counts == 2])

In [ ]:
counts = df["label"].value_counts()

# Keep only classes with at least 2 samples
valid_classes = counts[counts >= 2].index

df_filtered = df[df["label"].isin(valid_classes)].copy()

print("Original images :", len(df))
print("Filtered images :", len(df_filtered))

print("Original classes :", df["label"].nunique())
print("Filtered classes :", df_filtered["label"].nunique())

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df_filtered,
    test_size=0.2,
    random_state=42,
    stratify=df_filtered["label"],
)

print(len(train_df))
print(len(val_df))

In [ ]:
print(train_df["label"].value_counts().sort_index())
print(val_df["label"].value_counts().sort_index())

In [ ]:
from src.label_mapping import load_label_mapping

label_to_idx, idx_to_label = load_label_mapping("../artifacts")
NUM_CLASSES = len(label_to_idx)

print(NUM_CLASSES)

In [ ]:
# Mappings are saved centrally in artifacts/ directory. No local generation needed.

In [ ]:
class TankriCNN(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),


            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),


            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )

        )

        self.classifier = nn.Sequential(
            
            nn.AdaptiveAvgPool2d((1, 1)),

            nn.Flatten(),

            nn.Linear(
                128,
                num_classes
            )

        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x

In [ ]:
# Instantiate model dynamically using config fields
if config.MODEL_NAME == "ResNet18":
    from src.models import ResNet18Model
    model = ResNet18Model(
        num_classes=NUM_CLASSES,
        pretrained=config.PRETRAINED,
        dropout=config.DROPOUT,
        unfreeze_layer3=config.UNFREEZE_LAYER3,
        unfreeze_layer4=config.UNFREEZE_LAYER4,
    )
elif config.MODEL_NAME == "SimpleCNN":
    from src.models import SimpleCNN
    model = SimpleCNN(
        num_classes=NUM_CLASSES,
        dropout=config.DROPOUT,
    )
else:
    raise ValueError(f"Unknown model name: {config.MODEL_NAME}")

print(model)


In [ ]:
total_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Trainable Parameters: {total_params:,}")

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=config.LABEL_SMOOTHING)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config.LEARNING_RATE
)


In [ ]:
train_dataset = TankriDataset(
    dataframe=train_df,
    image_dir="../dataset/images",
    transform=transform
)

val_dataset = TankriDataset(
    dataframe=val_df,
    image_dir="../dataset/images",
    transform=transform
)

In [ ]:
BATCH_SIZE = config.BATCH_SIZE

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [ ]:
EPOCHS = config.EPOCHS

# Start an MLflow run
with mlflow.start_run(run_name=config.EXPERIMENT_NAME):

    # Configuration Logging (Snapshot and Parameters)
    from src.train import log_experiment_config, save_config_snapshot
    save_config_snapshot("config_snapshot.json")
    mlflow.log_artifact("config_snapshot.json")
    
    # Dynamically log all experiment config values
    log_experiment_config()

    # Automatically serialize the torchvision augmentation config and log it
    transform = getattr(train_loader.dataset, "transform", None)
    if transform is not None:
        from src.experiment_logging import save_augmentation_config
        aug_path = save_augmentation_config(transform, "augmentation_config.txt")
        mlflow.log_artifact(str(aug_path))

    train_losses = []
    train_accuracies = []

    val_losses = []
    val_accuracies = []

    best_val_acc = 0.0
    best_epoch = 0

    for epoch in range(EPOCHS):

        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        # Apply MixUp if configured
        from src.config import MIXUP_ALPHA
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            
            if MIXUP_ALPHA > 0.0:
                import numpy as np
                lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
                batch_size_curr = images.size()[0]
                index = torch.randperm(batch_size_curr).to(device)
                mixed_images = lam * images + (1 - lam) * images[index]
                labels_a, labels_b = labels, labels[index]
                
                outputs = model(mixed_images)
                loss = lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)
                loss.backward()
                optimizer.step()
                
                running_loss += loss.item()
                predictions = outputs.argmax(dim=1)
                correct += (lam * (predictions == labels_a).sum().item() + (1 - lam) * (predictions == labels_b).sum().item())
                total += labels.size(0)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                
                running_loss += loss.item()
                predictions = outputs.argmax(dim=1)
                correct += (predictions == labels).sum().item()
                total += labels.size(0)
                
        train_loss = running_loss / len(train_loader)
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        model.eval()
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_running_loss += loss.item()
                predictions = outputs.argmax(dim=1)
                val_correct += (predictions == labels).sum().item()
                val_total += labels.size(0)
        val_loss = val_running_loss / len(val_loader)
        val_acc = val_correct / val_total
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_accuracy", train_acc, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_accuracy", val_acc, step=epoch)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), "best_model.pth")

        print(
            f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

    mlflow.log_metric("best_val_accuracy", best_val_acc)
    mlflow.log_metric("best_epoch", best_epoch)
    mlflow.set_tag("best_epoch", str(best_epoch))
    mlflow.set_tag("best_val_accuracy", str(best_val_acc))

    # Log evaluation artifacts via the evaluation module
    history = {
        "train_loss": train_losses,
        "val_loss": val_losses,
        "train_accuracy": train_accuracies,
        "val_accuracy": val_accuracies,
    }
    from src.evaluate import log_evaluation_artifacts
    log_evaluation_artifacts(
        model=model,
        val_loader=val_loader,
        device=device,
        save_dir=".",
        history=history,
        label_to_idx=label_to_idx,
        idx_to_label=idx_to_label,
    )


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")

plt.legend()
plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Curve")

plt.legend()
plt.grid()

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np
model.load_state_dict(torch.load("best_model.pth"))

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)

        outputs = model(images)

        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
    
cm = confusion_matrix(
    all_labels,
    all_preds
)

print(cm.shape)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=[
            idx_to_label[i]
            for i in range(NUM_CLASSES)
        ],
        zero_division=0
    )
)